In [1]:
import json, os, csv
from collections import defaultdict, Counter
from tqdm import tqdm

BASE_DIR       = r'D:\Memoire\mooccubeX'
TRANSLATED_DIR = os.path.join(BASE_DIR, 'translated')
OUTPUT_DIR     = r'D:\Memoire\LLMKGER\kg'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def p(f):  return os.path.join(BASE_DIR, f)
def pt(f): return os.path.join(TRANSLATED_DIR, f)

print(f'Output dir : {OUTPUT_DIR}')


Output dir : D:\Memoire\LLMKGER\kg


In [2]:
courses = {}
with open(pt('course_translated.json'), encoding='utf-8', errors='replace') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try:
            obj = json.loads(line)
            cid = obj.get('id')
            if cid:
                courses[cid] = {'name': obj.get('name',''), 'field': obj.get('field',[]), 'about': obj.get('about','')}
        except: pass
print(f'Cours charges : {len(courses):,}')


Cours charges : 3,781


In [3]:
videos = {}
with open(pt('video_translated.json'), encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='Videos'):
        line = line.strip()
        if not line: continue
        try:
            obj = json.loads(line)
            ccid = obj.get('ccid')
            if ccid:
                videos[ccid] = {'name': obj.get('name','')}
        except: pass
print(f'Videos chargees : {len(videos):,}')


Videos: 59581it [00:09, 6134.64it/s]

Videos chargees : 59,581


In [4]:
concepts = {}
with open(pt('concept_translated.json'), encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='Concepts'):
        line = line.strip()
        if not line: continue
        try:
            obj = json.loads(line)
            kid = obj.get('id')
            if kid:
                concepts[kid] = {'name': obj.get('name','')}
        except: pass
print(f'Concepts charges : {len(concepts):,}')


Concepts: 637572it [00:04, 151332.07it/s]

Concepts charges : 637,572


In [5]:
teachers = {}
with open(pt('teacher_translated.json'), encoding='utf-8', errors='replace') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try:
            obj = json.loads(line)
            tid = obj.get('id')
            if tid:
                teachers[tid] = {'name': obj.get('name_en') or obj.get('name','')}
        except: pass
print(f'Teachers : {len(teachers):,}')

schools = {}
with open(pt('school_translated.json'), encoding='utf-8', errors='replace') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try:
            obj = json.loads(line)
            sid = obj.get('id')
            if sid:
                schools[sid] = {'name': obj.get('name_en') or obj.get('name','')}
        except: pass
print(f'Schools  : {len(schools):,}')


Teachers : 17,018
Schools  : 429


In [6]:
active_users = set()
with open(p('user-video.json'), encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='Users actifs'):
        line = line.strip()
        if not line: continue
        try:
            obj = json.loads(line)
            uid = obj.get('user_id')
            if uid:
                active_users.add(uid)
        except: pass
print(f'Users actifs : {len(active_users):,}')


Users actifs: 310360it [01:31, 3404.02it/s] 

Users actifs : 310,360


In [7]:
# Mapping V_id -> ccid (seulement ccids dans video_translated)
vid_to_ccid = {}
with open(p('video_id-ccid.txt'), encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='vid->ccid'):
        parts = line.strip().split('\t')
        if len(parts) == 2:
            vid, ccid = parts[0].strip(), parts[1].strip()
            if ccid in videos:
                vid_to_ccid[vid] = ccid
print(f'Mapping V_id->ccid : {len(vid_to_ccid):,}')


vid->ccid: 2798892it [00:03, 746153.98it/s]

Mapping V_id->ccid : 1,316,090


In [8]:
triples_watches = set()
with open(p('user-video.json'), encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='watches'):
        line = line.strip()
        if not line: continue
        try:
            obj = json.loads(line)
            uid = obj.get('user_id')
            if not uid: continue
            for entry in obj.get('seq', []):
                vid = entry.get('video_id')
                if vid and vid in vid_to_ccid:
                    triples_watches.add((uid, 'watches', vid_to_ccid[vid]))
        except: pass
print(f'Triplets watches (User->Video) : {len(triples_watches):,}')


watches: 310360it [02:42, 1913.70it/s] 

Triplets watches (User->Video) : 1,347,769


In [9]:
triples_has_concept = set()
with open(p('concept-video.txt'), encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='has_concept'):
        parts = line.strip().split('\t')
        if len(parts) == 2:
            kid, ccid = parts[0].strip(), parts[1].strip()
            if kid in concepts and ccid in videos:
                triples_has_concept.add((ccid, 'has_concept', kid))
print(f'Triplets has_concept (Video->Concept) : {len(triples_has_concept):,}')


has_concept: 624683it [00:01, 414740.06it/s]

Triplets has_concept (Video->Concept) : 624,683


In [10]:
triples_belongs_to = set()
course_to_concepts_ordered = defaultdict(list)
with open(p('concept-course.txt'), encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='belongs_to'):
        parts = line.strip().split('\t')
        if len(parts) == 2:
            kid, cid = parts[0].strip(), parts[1].strip()
            if kid in concepts and cid in courses:
                triples_belongs_to.add((kid, 'belongs_to', cid))
                course_to_concepts_ordered[cid].append(kid)
print(f'Triplets belongs_to (Concept->Course) : {len(triples_belongs_to):,}')
print(f'Cours avec concepts : {len(course_to_concepts_ordered):,}')


belongs_to: 451078it [00:01, 374202.75it/s]

Triplets belongs_to (Concept->Course) : 451,078
Cours avec concepts : 887


In [11]:
triples_taught_by = set()
with open(p('course-teacher.txt'), encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='taught_by'):
        parts = line.strip().split('\t')
        if len(parts) == 2:
            cid, tid = parts[0].strip(), parts[1].strip()
            if cid in courses and tid in teachers:
                triples_taught_by.add((cid, 'taught_by', tid))
print(f'Triplets taught_by  : {len(triples_taught_by):,}')

triples_offered_by = set()
with open(p('course-school.txt'), encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='offered_by'):
        parts = line.strip().split('\t')
        if len(parts) == 2:
            cid, sid = parts[0].strip(), parts[1].strip()
            if cid in courses and sid in schools:
                triples_offered_by.add((cid, 'offered_by', sid))
print(f'Triplets offered_by : {len(triples_offered_by):,}')


taught_by: 97192it [00:00, 637587.11it/s]


Triplets taught_by  : 13,312


offered_by: 3983it [00:00, 422870.27it/s]

Triplets offered_by : 3,605


In [12]:
triples_follows = set()
skipped = 0
with open(p('user.json'), encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='follows'):
        line = line.strip()
        if not line: continue
        try:
            obj = json.loads(line)
            uid = obj.get('id')
            if not uid: continue
            for order in obj.get('course_order', []):
                cid = f'C_{order}'
                if cid in courses:
                    triples_follows.add((uid, 'follows', cid))
                else:
                    skipped += 1
        except: pass
print(f'Triplets follows (User->Course) : {len(triples_follows):,}')
print(f'Orders ignores (cours absents)  : {skipped:,}')


follows: 3330294it [00:36, 90722.71it/s] 

Triplets follows (User->Course) : 11,523,119
Orders ignores (cours absents)  : 283,971


In [13]:
MIN_COUNT = 3
WINDOW    = 3
co_occ = Counter()
for cid, concept_list in tqdm(course_to_concepts_ordered.items(), desc='prereq'):
    for i in range(len(concept_list)):
        for j in range(i+1, min(i+1+WINDOW, len(concept_list))):
            ca, cb = concept_list[i], concept_list[j]
            if ca != cb:
                co_occ[(ca, cb)] += 1

triples_prereq = set()
for (ca, cb), cnt in co_occ.items():
    if cnt >= MIN_COUNT:
        triples_prereq.add((ca, 'prerequisite_of', cb))

print(f'Paires brutes generees          : {len(co_occ):,}')
print(f'Triplets prerequisite_of (>={MIN_COUNT}) : {len(triples_prereq):,}')
print(f'Exemples : {list(triples_prereq)[:3]}')


prereq: 100%|██████████| 887/887 [00:01<00:00, 638.78it/s]


Paires brutes generees          : 887,016
Triplets prerequisite_of (>=3) : 84,968
Exemples : [('K_存放路径_计算机科学与技术', 'prerequisite_of', 'K_http后端服务器的架构_计算机科学与技术'), ('K_提示_计算机科学与技术', 'prerequisite_of', 'K_差_计算机科学与技术'), ('K_任务_计算机科学与技术', 'prerequisite_of', 'K_乱序执行_计算机科学与技术')]


In [14]:
all_relations = {
    'watches'        : triples_watches,
    'has_concept'    : triples_has_concept,
    'belongs_to'     : triples_belongs_to,
    'taught_by'      : triples_taught_by,
    'offered_by'     : triples_offered_by,
    'follows'        : triples_follows,
    'prerequisite_of': triples_prereq,
}
total_triples = sum(len(v) for v in all_relations.values())

print('=' * 50)
print('  KG LLMKGER — Statistiques')
print('=' * 50)
print('\nNOEUDS :')
for label, cnt in [('Users (actifs)', len(active_users)), ('Courses', len(courses)),
                    ('Videos (ccid)', len(videos)), ('Concepts', len(concepts)),
                    ('Teachers', len(teachers)), ('Schools', len(schools))]:
    print(f'  {label:<20} {cnt:>10,}')
print('\nARETES :')
for rel, triples in all_relations.items():
    print(f'  {rel:<20} {len(triples):>10,}')
print(f'  {"TOTAL":<20} {total_triples:>10,}')


  KG LLMKGER — Statistiques

NOEUDS :
  Users (actifs)          310,360
  Courses                   3,781
  Videos (ccid)            59,581
  Concepts                637,572
  Teachers                 17,018
  Schools                     429

ARETES :
  watches               1,347,769
  has_concept             624,683
  belongs_to              451,078
  taught_by                13,312
  offered_by                3,605
  follows              11,523,119
  prerequisite_of          84,968
  TOTAL                14,048,534


In [15]:
# Export triplets.tsv
with open(os.path.join(OUTPUT_DIR, 'triplets.tsv'), 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='\t')
    w.writerow(['head', 'relation', 'tail'])
    for rel, triples in all_relations.items():
        for h, r, t in triples:
            w.writerow([h, r, t])
print('triplets.tsv exporte')

# Export entities.tsv
with open(os.path.join(OUTPUT_DIR, 'entities.tsv'), 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='\t')
    w.writerow(['id', 'type', 'name'])
    for eid, info in courses.items():   w.writerow([eid, 'course',  info['name']])
    for eid, info in videos.items():    w.writerow([eid, 'video',   info['name']])
    for eid, info in concepts.items():  w.writerow([eid, 'concept', info['name']])
    for eid, info in teachers.items():  w.writerow([eid, 'teacher', info['name']])
    for eid, info in schools.items():   w.writerow([eid, 'school',  info['name']])
    for uid in active_users:            w.writerow([uid, 'user',    ''])
print('entities.tsv exporte')

# Export relations.tsv
meta = {'watches':('user','video'), 'has_concept':('video','concept'),
        'belongs_to':('concept','course'), 'taught_by':('course','teacher'),
        'offered_by':('course','school'), 'follows':('user','course'),
        'prerequisite_of':('concept','concept')}
with open(os.path.join(OUTPUT_DIR, 'relations.tsv'), 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='\t')
    w.writerow(['relation', 'count', 'head_type', 'tail_type'])
    for rel, triples in all_relations.items():
        h, t = meta[rel]
        w.writerow([rel, len(triples), h, t])
print('relations.tsv exporte')

# Export user_course_interactions.tsv
with open(os.path.join(OUTPUT_DIR, 'user_course_interactions.tsv'), 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='\t')
    w.writerow(['user_id', 'course_id'])
    for u, _, c in triples_follows:
        w.writerow([u, c])
print('user_course_interactions.tsv exporte')

print('\nFichiers dans', OUTPUT_DIR)
for fname in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size  = round(os.path.getsize(fpath)/1024/1024, 1)
    print(f'  {fname:<42} {size} MB')


triplets.tsv exporte
entities.tsv exporte
relations.tsv exporte
user_course_interactions.tsv exporte

Fichiers dans D:\Memoire\LLMKGER\kg
  entities.tsv                               47.2 MB
  relations.tsv                              0.0 MB
  triplets.tsv                               463.8 MB
  user_course_interactions.tsv               230.4 MB
